In [1]:
#Source for tutorial: https://docs.dgl.ai/guide/training-node.html

import dgl.nn as dglnn
import torch.nn as nn
import torch.nn.functional as F
class SAGE(nn.Module):
    def __init__(self, in_feats, hid_feats, out_feats):
        super().__init__()
        self.conv1 = dglnn.SAGEConv(
            in_feats=in_feats, out_feats=hid_feats, aggregator_type='mean')
        self.conv2 = dglnn.SAGEConv(
            in_feats=hid_feats, out_feats=out_feats, aggregator_type='mean')

    def forward(self, graph, inputs):
        # inputs are features of nodes
        h = self.conv1(graph, inputs)
        h = F.relu(h)
        h = self.conv2(graph, h)
        return h

Using backend: pytorch


In [38]:
from dgl.data import QM9Dataset



# Can expand this to get multiple values per node
data = QM9Dataset(label_keys=['mu'], cutoff=5.0)
graph = data[0]
print("Number of graphs: ", round(data.__len__()*2/3))
num_train_graphs = round(data.__len__()*2/3)

train = data[:num_train_graphs]
test = data[num_train_graphs:data.__len__()]
# iterate over the dataset
# for g, label in data:
#     # Properties of each graph node
#     R = g.ndata['R'] # get coordinates of each atom
#     Z = g.ndata['Z'] # get atomic numbers of each atom
    
    
# node_features = graph.ndata['feat']
# node_labels = graph.ndata['label']
# train_mask = graph.ndata['train_mask']
# valid_mask = graph.ndata['val_mask']
# test_mask = graph.ndata['test_mask']
# n_features = node_features.shape[1]
# n_labels = int(node_labels.max().item() + 1)

ImportError: cannot import name 'QM9EdgeDataset' from 'dgl.data' (C:\Users\wings\anaconda3\lib\site-packages\dgl\data\__init__.py)

In [ ]:
def evaluate(model, graph, features, labels, mask):
    model.eval()
    with torch.no_grad():
        logits = model(graph, features)
        logits = logits[mask]
        labels = labels[mask]
        _, indices = torch.max(logits, dim=1)
        correct = torch.sum(indices == labels)
        return correct.item() * 1.0 / len(labels)

In [5]:
# Training Model
import torch

model = SAGE(in_feats=n_features, hid_feats=100, out_feats=n_labels)
opt = torch.optim.Adam(model.parameters())

for epoch in range(10):
    model.train()
    # forward propagation by using all nodes
    logits = model(graph, node_features)
    print(logits)
    # compute loss
    loss = F.cross_entropy(logits[train_mask], node_labels[train_mask])
    # compute validation accuracy
    acc = evaluate(model, graph, node_features, node_labels, valid_mask)
    # backward propagation
    opt.zero_grad()
    loss.backward()
    opt.step()

    # Save model if necessary.  Omitted in this example.

tensor([[ 0.0051,  0.0851,  0.1222,  0.0021, -0.1000, -0.1777],
        [-0.0167,  0.1139,  0.1039,  0.0148, -0.0744, -0.1416],
        [-0.0164,  0.1090,  0.1065,  0.0185, -0.0565, -0.1877],
        ...,
        [-0.0286,  0.1110,  0.1006,  0.0118, -0.0850, -0.1569],
        [-0.0501,  0.1157,  0.1267,  0.0455, -0.0662, -0.1620],
        [-0.0134,  0.1113,  0.1060,  0.0145, -0.0754, -0.1529]],
       grad_fn=<AddBackward0>)


C:\Users\wings\anaconda3\lib\site-packages\torch\autograd\__init__.py:130: UserWarning: CUDA initialization: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx (Triggered internally at  ..\c10\cuda\CUDAFunctions.cpp:100.)
  Variable._execution_engine.run_backward(


tensor([[ 0.0034,  0.0633,  0.1062,  0.0197, -0.0981, -0.1698],
        [-0.0110,  0.1074,  0.0924,  0.0068, -0.0717, -0.1322],
        [-0.0143,  0.0880,  0.0904,  0.0157, -0.0556, -0.1578],
        ...,
        [-0.0224,  0.0948,  0.0893,  0.0112, -0.0814, -0.1489],
        [-0.0422,  0.1012,  0.1142,  0.0391, -0.0632, -0.1496],
        [-0.0089,  0.0930,  0.0901,  0.0056, -0.0701, -0.1383]],
       grad_fn=<AddBackward0>)
tensor([[-0.0030,  0.0396,  0.0911,  0.0371, -0.0979, -0.1635],
        [-0.0110,  0.1008,  0.0805, -0.0003, -0.0695, -0.1221],
        [-0.0171,  0.0650,  0.0759,  0.0136, -0.0554, -0.1303],
        ...,
        [-0.0211,  0.0778,  0.0784,  0.0125, -0.0800, -0.1404],
        [-0.0381,  0.0842,  0.0995,  0.0335, -0.0615, -0.1370],
        [-0.0075,  0.0757,  0.0769,  0.0018, -0.0664, -0.1256]],
       grad_fn=<AddBackward0>)
tensor([[-0.0115,  0.0197,  0.0782,  0.0551, -0.1001, -0.1575],
        [-0.0123,  0.1009,  0.0699, -0.0080, -0.0701, -0.1105],
        [-0.02

In [6]:
# print(node_labels.sum().item())
# print(train_mask.size())
# print(test_mask.size())
# print(valid_mask.size())


model.eval()
# Convert a list of tuples to two lists
test_X, test_Y = map(list, zip(*testset))
test_bg = dgl.batch(test_X)
test_Y = torch.tensor(test_Y).float().view(-1, 1)
#probs_Y = torch.softmax(model(test_bg), 1)
probs_Y = model(test_bg)
print(probs_Y)
# test_X, test_Y = map(list, zip(*testset))
# print(test_Y)
# print(model)
# exit()
# sampled_Y = torch.multinomial(probs_Y, 1)
# argmax_Y = torch.max(probs_Y, 1)[1].view(-1, 1)
# print('Accuracy of sampled predictions on the test set: {:.4f}%'.format(
#     (test_Y == sampled_Y.float()).sum().item() / len(test_Y) * 100))
# print('Accuracy of argmax predictions on the test set: {:4f}%'.format(
#     (test_Y == argmax_Y.float()).sum().item() / len(test_Y) * 100))

NameError: name 'testset' is not defined